# Sequential FreeGSNKE ↔ TORAX Coupling

**Self-consistent equilibrium-transport coupling for pre-disruption plasma evolution.**

## Algorithm

At each 1-second coupling interval:
1. Extract betap from TORAX's previous output profiles
2. FreeGSNKE: `solve_static(Ip, betap)` → write GEQDSK
3. TORAX: run 1-second mini-trajectory with that GEQDSK (geometry frozen during interval)
4. Check disruption triggers: βN > 3.2, q95 < 2.2, f_GW > 0.95
5. If trigger fires → record pre-disruption state for DREAM handoff

## Expected runtime
- First TORAX call: ~2 min (JAX compilation)
- Subsequent calls: ~30-60s each
- 10-step run: ~10 min total

## Three scenarios
1. **Normal shot** — constant Ip, heating, density (validates coupling stability)
2. **Density-limit disruption** — ramp nbar until f_GW > 0.95
3. **q95 disruption** — ramp Ip from 15 MA → 8 MA until q95 < 2.2

In [ ]:
# ============================================================
# INSTALL CELL — installs packages if missing.
# Safe to re-run: skips if already installed.
# ============================================================

import importlib.util, subprocess, re, os, warnings
warnings.filterwarnings("ignore", message="JAX plugin")

def _pip(*args):
    subprocess.run(["pip", "install", "-q"] + list(args), check=False)

need_torax = importlib.util.find_spec("torax") is None
need_freegs = importlib.util.find_spec("freegsnke") is None

if need_torax or need_freegs:
    print("Installing packages...")

    if need_torax:
        print("  [1/5] Installing TORAX + JAX...")
        _pip("torax")

    print("  [2/5] Installing JAX CUDA support...")
    _pip("jax[cuda12]")

    if need_freegs:
        print("  [3/5] Installing FreeGSNKE (bypassing numpy pin)...")
        _pip("freegs4e", "--no-deps")
        _pip("freegsnke", "--no-deps")

    print("  [4/5] Installing scipy / matplotlib / h5py / xarray / deepdiff...")
    _pip("scipy", "matplotlib", "h5py", "netCDF4", "xarray", "deepdiff")

    r = subprocess.run(["pip", "show", "numpy"], capture_output=True, text=True)
    m = re.search(r"Version: ([\d.]+)", r.stdout)
    nv = m.group(1) if m else "2.4.3"
    print(f"  [5/5] Force-reinstalling numpy {nv} to fix ABI...")
    _pip(f"numpy=={nv}", "--force-reinstall", "--no-deps")

    subprocess.run(["pip", "uninstall", "numba", "-y", "-q"], check=False)

    print("\nDone. Restarting kernel — re-run all cells after reconnect.")
    os.kill(os.getpid(), 9)

else:
    import numpy as np, jax
    print(f"Packages already installed.")
    print(f"  numpy  : {np.__version__}")
    print(f"  jax    : {jax.__version__}")

In [ ]:
import numpy as np

# Numpy 2.x compatibility patch for freegs4e/freegsnke
for _name, _builtin in [('float', float), ('int', int),
                         ('bool', bool), ('complex', complex),
                         ('object', object), ('str', str)]:
    if not hasattr(np, _name):
        setattr(np, _name, _builtin)

# Fake numba module — prevents freegs4e crash
import types, sys
_fake_numba = types.ModuleType("numba")
def _njit(*args, **kwargs):
    if args and callable(args[0]):
        return args[0]
    def _decorator(func):
        return func
    return _decorator
_fake_numba.njit = _njit
sys.modules["numba"] = _fake_numba

import freegsnke, freegs4e, jax, torax, h5py

print(f"numpy     : {np.__version__}")
print(f"freegsnke : {freegsnke.__version__}")
print(f"jax       : {jax.__version__}")
print(f"torax     : {torax.__version__}")
print(f"JAX devices: {jax.devices()}")

# Clone repos
import os
REPO_URL = "https://github.com/BecerraMiguel/tokamak-disruption-simulator.git"
REPO_DIR = "/content/tokamak-disruption-simulator"
DINA_URL = "https://github.com/iterorganization/DINA-IMAS.git"
DINA_DIR = "/content/DINA-IMAS"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} && git pull")

if not os.path.exists(DINA_DIR):
    os.system(f"git clone --no-checkout --depth=1 {DINA_URL} {DINA_DIR}")
    os.system(f"cd {DINA_DIR} && git sparse-checkout init --cone")
    os.system(f"cd {DINA_DIR} && git sparse-checkout set machines")
    os.system(f"cd {DINA_DIR} && git checkout")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))

assert os.path.exists(f"{DINA_DIR}/machines/iter/tokamak_config.dat"), \
    "ITER config not found"
print("Repository ready.")

In [ ]:
from predisruption.iter_machine import build_iter_machine, ITER_PARAMS
from predisruption.equilibrium import EquilibriumSolver

CONFIG_PATH = f"{DINA_DIR}/machines/iter/tokamak_config.dat"

tokamak, active_coils, domain = build_iter_machine(
    config_path=CONFIG_PATH,
    include_passives=False,
    verbose=True,
)

eq_solver = EquilibriumSolver(
    tokamak=tokamak,
    nx=65, ny=65,
    Rmin=domain[0], Rmax=domain[1],
    Zmin=domain[2], Zmax=domain[3],
    verbose=True,
)

print("Solving initial 15 MA equilibrium...")
eq = eq_solver.solve_static(Ip=15.0e6, betap=0.5)
signals = eq_solver.get_signals(eq)

print(f"\nInitial equilibrium:")
print(f"  Ip    = {signals['Ip']*1e-6:.2f} MA")
print(f"  q95   = {signals['q95']:.2f}")
print(f"  betaN = {signals['betaN']:.3f}")
print(f"  betap = {signals['betap']:.3f}")
print(f"  kappa = {signals['kappa']:.2f}")
print(f"  delta = {signals['delta']:.2f}")
print(f"  a_min = {signals['a_minor']:.3f} m")

In [ ]:
import copy
import tempfile
import gc
import time as time_mod
import matplotlib.pyplot as plt
from predisruption.transport import _default_iter_torax_config, _deep_merge

# ---------------------------------------------------------------------------
# Trigger thresholds (same as shot_runner.py)
# ---------------------------------------------------------------------------
TRIGGERS = {
    "f_GW":  0.95,
    "betaN": 3.2,
    "q95":   2.2,
}

# ---------------------------------------------------------------------------
# Extract final profiles from a TORAX DataTree
# ---------------------------------------------------------------------------
def extract_torax_final_profiles(data_tree):
    """Extract final-timestep profiles from TORAX DataTree output.

    Handles rho grid mismatch: each variable has its own rho coordinate
    (rho_cell_norm vs rho_face_norm).
    """
    profiles = data_tree["profiles"].ds
    scalars = data_tree["scalars"].ds

    def _get(var_name):
        dim = profiles[var_name].dims[-1]
        rho = profiles.coords[dim].values
        vals = profiles[var_name].values[-1, :]
        return rho, vals

    rho_Te, Te = _get("T_e")
    rho_Ti, Ti = _get("T_i")
    rho_ne, ne = _get("n_e")
    rho_j, j = _get("j_total")

    if "psi" in profiles:
        rho_psi, psi = _get("psi")
    else:
        rho_psi, psi = rho_Te, np.zeros_like(Te)

    if "q" in profiles:
        rho_q, q = _get("q")
    else:
        rho_q, q = rho_Te, np.ones_like(Te) * 3.0

    W_th = float(scalars["W_thermal_total"].values[-1]) if "W_thermal_total" in scalars else 0.0
    Ip_torax = float(scalars["Ip"].values[-1]) if "Ip" in scalars else None

    return {
        "Te": Te, "rho_Te": rho_Te,
        "Ti": Ti, "rho_Ti": rho_Ti,
        "ne": ne, "rho_ne": rho_ne,
        "j": j, "rho_j": rho_j,
        "psi": psi, "rho_psi": rho_psi,
        "q": q, "rho_q": rho_q,
        "W_thermal": W_th,
        "Ip": Ip_torax,
    }


# ---------------------------------------------------------------------------
# Build TORAX config for one sequential mini-run
# ---------------------------------------------------------------------------
def build_sequential_torax_config(
    geqdsk_dir,
    geqdsk_file,
    dt_couple,
    Ip_A,
    P_heat_W,
    nbar=0.85,
    transport_model="constant",
    initial_profiles=None,
):
    """Build a TORAX config dict for a single coupling interval.

    Parameters
    ----------
    geqdsk_dir : str
        Directory containing the GEQDSK file.
    geqdsk_file : str
        Filename of the GEQDSK for this step's geometry.
    dt_couple : float
        Duration of this mini-run (seconds).
    Ip_A : float
        Plasma current (A).
    P_heat_W : float
        Total heating power (W).
    nbar : float
        Line-averaged density in Greenwald fraction units.
    transport_model : str
        TORAX transport model ('constant', 'bohm-gyrobohm', 'qlknn').
    initial_profiles : dict or None
        Output from extract_torax_final_profiles(). If provided, sets the
        initial condition from the previous TORAX run for profile continuity.
    """
    config = _default_iter_torax_config(
        geometry_dir=geqdsk_dir,
        geometry_files={0.0: geqdsk_file},
        t_final=dt_couple,
        Ip_A=Ip_A,
        P_heat_W=P_heat_W,
        transport_model=transport_model,
    )

    # Override nbar
    config["profile_conditions"]["nbar"] = nbar

    # If we have profiles from the previous step, use them as initial conditions
    if initial_profiles is not None:
        pc = config["profile_conditions"]

        def _profile_to_tva(rho, vals):
            """Convert (rho_array, value_array) to {0.0: {rho[i]: val[i]}} dict."""
            return {0.0: {float(rho[i]): float(vals[i]) for i in range(len(rho))}}

        # Temperature profiles
        pc["T_e"] = _profile_to_tva(initial_profiles["rho_Te"], initial_profiles["Te"])
        pc["T_i"] = _profile_to_tva(initial_profiles["rho_Ti"], initial_profiles["Ti"])

        # Temperature boundary conditions from edge values
        pc["T_e_right_bc"] = max(float(initial_profiles["Te"][-1]), 0.05)
        pc["T_i_right_bc"] = max(float(initial_profiles["Ti"][-1]), 0.05)

        # Density profile — provide profile shape, normalize to nbar target
        pc["n_e"] = _profile_to_tva(initial_profiles["rho_ne"], initial_profiles["ne"])
        pc["normalize_n_e_to_nbar"] = True
        pc["n_e_nbar_is_fGW"] = True
        pc["nbar"] = nbar

        # Psi profile for current continuity
        if initial_profiles.get("psi") is not None and np.any(initial_profiles["psi"] != 0):
            pc["psi"] = _profile_to_tva(initial_profiles["rho_psi"], initial_profiles["psi"])
            pc["initial_psi_mode"] = "profile_conditions"

    return config


# ---------------------------------------------------------------------------
# Compute betap from TORAX profiles
# ---------------------------------------------------------------------------
def compute_betap_from_profiles(Te_keV, Ti_keV, ne_m3, Ip_A, a_minor):
    """Compute poloidal beta from TORAX output profiles.

    CRITICAL: 1 keV = 1.602e-16 J (NOT 1.602e-16 * 1e3).
    """
    mu0 = 4e-7 * np.pi
    T_e_J = Te_keV * 1.602e-16   # keV -> J
    T_i_J = Ti_keV * 1.602e-16
    p_avg = float(np.mean(ne_m3 * (T_e_J + T_i_J)))  # Pa

    B_p = mu0 * Ip_A / (2.0 * np.pi * a_minor)
    betap = p_avg / (B_p**2 / (2.0 * mu0))

    return max(betap, 0.01)


# ---------------------------------------------------------------------------
# Check disruption triggers
# ---------------------------------------------------------------------------
def check_triggers(eq_signals, ne_profile, Ip_A, a_minor):
    """Check disruption trigger conditions.

    Returns (triggered, trigger_name, values_dict).
    """
    Ip_MA = abs(Ip_A) * 1e-6
    n_GW = Ip_MA / (np.pi * a_minor**2) * 1e20  # m^-3
    nbar = float(np.mean(ne_profile))
    f_GW = nbar / n_GW if n_GW > 0 else 0.0

    betaN = eq_signals.get("betaN", 0.0)
    q95 = eq_signals.get("q95", 10.0)

    values = {"f_GW": f_GW, "betaN": betaN, "q95": q95}

    if f_GW > TRIGGERS["f_GW"]:
        return True, "f_GW", values
    if betaN > TRIGGERS["betaN"]:
        return True, "betaN", values
    if q95 < TRIGGERS["q95"]:
        return True, "q95", values

    return False, None, values


print("Helper functions defined.")

In [ ]:
def run_sequential_coupling(
    eq_solver,
    t_end=10.0,
    dt_couple=1.0,
    Ip_waveform=lambda t: 15e6,
    P_heat_waveform=lambda t: 33e6,
    nbar_waveform=lambda t: 0.85,
    transport_model="constant",
    verbose=True,
):
    """Sequential FreeGSNKE <-> TORAX coupling.

    Alternates between FreeGSNKE equilibrium solves and TORAX 1-second
    mini-trajectory runs. TORAX profiles feed back into FreeGSNKE via betap.

    Parameters
    ----------
    eq_solver : EquilibriumSolver
    t_end : float
        Total simulation time (s).
    dt_couple : float
        Coupling interval (s).
    Ip_waveform, P_heat_waveform, nbar_waveform : callable
        Time-dependent waveforms.
    transport_model : str
        TORAX transport model.
    verbose : bool

    Returns
    -------
    dict with trajectory data.
    """
    import torax

    n_steps = int(t_end / dt_couple)
    geqdsk_dir = tempfile.mkdtemp(prefix="seq_coupling_")

    # Storage
    times = []
    betap_hist = []
    betaN_hist = []
    q95_hist = []
    f_GW_hist = []
    W_thermal_hist = []
    Ip_hist = []
    Te_profiles = []
    ne_profiles = []
    j_profiles = []
    q_profiles = []
    rho_grid = None

    disruption_time = None
    trigger_name = None
    pre_disruption_profiles = None

    # Step 0: Initial equilibrium
    Ip_0 = Ip_waveform(0.0)
    betap_current = 0.5

    if verbose:
        print("=" * 60)
        print("Sequential FreeGSNKE <-> TORAX Coupling")
        print(f"  t_end={t_end}s, dt_couple={dt_couple}s, n_steps={n_steps}")
        print(f"  transport_model={transport_model}")
        print("=" * 60)

    eq = eq_solver.solve_static(Ip=Ip_0, betap=betap_current)
    eq_signals = eq_solver.get_signals(eq)
    a_minor = eq_signals.get("a_minor", ITER_PARAMS["a_minor"])

    geqdsk_file = "eq_step_0000.geqdsk"
    eq_solver.write_geqdsk(os.path.join(geqdsk_dir, geqdsk_file), eq)

    if verbose:
        print(f"\nInitial eq: Ip={eq_signals['Ip']*1e-6:.2f}MA, "
              f"q95={eq_signals['q95']:.2f}, betap={eq_signals['betap']:.3f}")

    prev_profiles = None

    for step in range(n_steps):
        t = step * dt_couple
        t_wall_start = time_mod.time()

        Ip_now = Ip_waveform(t)
        P_heat_now = P_heat_waveform(t)
        nbar_now = nbar_waveform(t)

        if verbose:
            print(f"\n--- Step {step}/{n_steps-1} | t={t:.1f}s "
                  f"| Ip={Ip_now*1e-6:.1f}MA | P={P_heat_now*1e-6:.1f}MW "
                  f"| nbar={nbar_now:.3f} ---")

        # 1. Build TORAX config
        config = build_sequential_torax_config(
            geqdsk_dir=geqdsk_dir,
            geqdsk_file=geqdsk_file,
            dt_couple=dt_couple,
            Ip_A=Ip_now,
            P_heat_W=P_heat_now,
            nbar=nbar_now,
            transport_model=transport_model,
            initial_profiles=prev_profiles,
        )

        # 2. Run TORAX mini-trajectory
        try:
            torax_config = torax.ToraxConfig.from_dict(config)
            data_tree, state_history = torax.run_simulation(torax_config)
        except Exception as e:
            print(f"  TORAX failed: {e}")
            print("  Skipping this step, reusing previous profiles.")
            times.append(t + dt_couple)
            betap_hist.append(betap_current)
            betaN_hist.append(eq_signals.get("betaN", 0.0))
            q95_hist.append(eq_signals.get("q95", 0.0))
            f_GW_hist.append(0.0)
            W_thermal_hist.append(0.0)
            Ip_hist.append(Ip_now)
            continue

        # 3. Extract final profiles
        prev_profiles = extract_torax_final_profiles(data_tree)

        # 4. Compute betap from TORAX profiles
        betap_new = compute_betap_from_profiles(
            Te_keV=prev_profiles["Te"],
            Ti_keV=prev_profiles["Ti"],
            ne_m3=prev_profiles["ne"],
            Ip_A=Ip_now,
            a_minor=a_minor,
        )
        betap_current = betap_new

        # Record scalars
        times.append(t + dt_couple)
        betap_hist.append(betap_current)
        Ip_hist.append(Ip_now)
        W_thermal_hist.append(prev_profiles["W_thermal"])

        # Store profiles on common grid
        if rho_grid is None:
            rho_grid = prev_profiles["rho_Te"].copy()
        Te_profiles.append(np.interp(rho_grid, prev_profiles["rho_Te"], prev_profiles["Te"]))
        ne_profiles.append(np.interp(rho_grid, prev_profiles["rho_ne"], prev_profiles["ne"]))
        j_profiles.append(np.interp(rho_grid, prev_profiles["rho_j"], prev_profiles["j"]))
        q_profiles.append(np.interp(rho_grid, prev_profiles["rho_q"], prev_profiles["q"]))

        # Free TORAX memory
        del data_tree, state_history
        if step % 5 == 0:
            gc.collect()
            try:
                jax.clear_caches()
            except Exception:
                pass

        # 5. FreeGSNKE: re-solve equilibrium with updated betap
        try:
            eq = eq_solver.solve_static(Ip=Ip_now, betap=betap_current)
            eq_signals = eq_solver.get_signals(eq)
            a_minor = eq_signals.get("a_minor", a_minor)
        except Exception as e:
            if verbose:
                print(f"  FreeGSNKE failed: {e} \u2014 reusing previous equilibrium")

        betaN_hist.append(eq_signals.get("betaN", 0.0))
        q95_hist.append(eq_signals.get("q95", 0.0))

        # 6. Check triggers
        triggered, trig_name, trig_vals = check_triggers(
            eq_signals, prev_profiles["ne"], Ip_now, a_minor,
        )
        f_GW_hist.append(trig_vals["f_GW"])

        t_wall = time_mod.time() - t_wall_start
        if verbose:
            print(f"  betap={betap_current:.3f}, betaN={trig_vals['betaN']:.3f}, "
                  f"q95={trig_vals['q95']:.2f}, f_GW={trig_vals['f_GW']:.3f} "
                  f"[{t_wall:.1f}s wall]")

        if triggered:
            disruption_time = t + dt_couple
            trigger_name = trig_name
            pre_disruption_profiles = prev_profiles
            if verbose:
                print(f"\n*** DISRUPTION TRIGGERED: {trig_name} "
                      f"at t={disruption_time:.1f}s ***")
                print(f"    Values: {trig_vals}")
            break

        # 7. Write new GEQDSK for next TORAX run
        geqdsk_file = f"eq_step_{step+1:04d}.geqdsk"
        eq_solver.write_geqdsk(os.path.join(geqdsk_dir, geqdsk_file), eq)

    # Build trajectory dict
    trajectory = {
        "time": np.array(times),
        "Ip": np.array(Ip_hist),
        "betap": np.array(betap_hist),
        "betaN": np.array(betaN_hist),
        "q95": np.array(q95_hist),
        "f_GW": np.array(f_GW_hist),
        "W_thermal": np.array(W_thermal_hist),
        "rho": rho_grid,
        "T_e": np.array(Te_profiles) if Te_profiles else np.array([]),
        "n_e": np.array(ne_profiles) if ne_profiles else np.array([]),
        "j_tor": np.array(j_profiles) if j_profiles else np.array([]),
        "q_profile": np.array(q_profiles) if q_profiles else np.array([]),
        "disruption_time": disruption_time,
        "trigger": trigger_name,
        "pre_disruption_profiles": pre_disruption_profiles,
        "geqdsk_dir": geqdsk_dir,
    }

    if verbose:
        status = f"DISRUPTION ({trigger_name})" if disruption_time else "COMPLETED"
        print(f"\n{'=' * 60}")
        print(f"Simulation {status}. {len(times)} steps recorded.")
        print(f"GEQDSK files in: {geqdsk_dir}")

    return trajectory


print("Sequential coupling function defined.")

## Scenario 1: Normal Shot (Validation)

Constant Ip=15 MA, P_heat=33 MW, nbar=0.85 fGW. Should reach steady state without triggering any disruption.

In [ ]:
print("Running normal shot (10s, constant params)...")
print("First TORAX call will take ~2 min for JAX compilation.\n")

trajectory_normal = run_sequential_coupling(
    eq_solver,
    t_end=10.0,
    dt_couple=1.0,
    Ip_waveform=lambda t: 15e6,
    P_heat_waveform=lambda t: 33e6,
    nbar_waveform=lambda t: 0.85,
    transport_model="constant",
    verbose=True,
)

# Summary table
print("\n\nStep Summary:")
print(f"{'t(s)':>6} {'betap':>8} {'betaN':>8} {'q95':>8} {'f_GW':>8} {'W_th(MJ)':>10}")
print("-" * 52)
for i in range(len(trajectory_normal["time"])):
    print(f"{trajectory_normal['time'][i]:6.1f} "
          f"{trajectory_normal['betap'][i]:8.3f} "
          f"{trajectory_normal['betaN'][i]:8.3f} "
          f"{trajectory_normal['q95'][i]:8.2f} "
          f"{trajectory_normal['f_GW'][i]:8.3f} "
          f"{trajectory_normal['W_thermal'][i]*1e-6:10.1f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

traj = trajectory_normal
t = traj["time"]

# betaN vs time
ax = axes[0, 0]
ax.plot(t, traj["betaN"], "b-o", markersize=4)
ax.axhline(TRIGGERS["betaN"], color="r", linestyle="--", alpha=0.7,
           label=f"limit={TRIGGERS['betaN']}")
ax.set_xlabel("Time (s)"); ax.set_ylabel("betaN")
ax.set_title("Normalized Beta"); ax.legend(); ax.grid(True, alpha=0.3)

# q95 vs time
ax = axes[0, 1]
ax.plot(t, traj["q95"], "g-o", markersize=4)
ax.axhline(TRIGGERS["q95"], color="r", linestyle="--", alpha=0.7,
           label=f"limit={TRIGGERS['q95']}")
ax.set_xlabel("Time (s)"); ax.set_ylabel("q95")
ax.set_title("Safety Factor at 95%"); ax.legend(); ax.grid(True, alpha=0.3)

# f_GW vs time
ax = axes[0, 2]
ax.plot(t, traj["f_GW"], "m-o", markersize=4)
ax.axhline(TRIGGERS["f_GW"], color="r", linestyle="--", alpha=0.7,
           label=f"limit={TRIGGERS['f_GW']}")
ax.set_xlabel("Time (s)"); ax.set_ylabel("f_GW")
ax.set_title("Greenwald Fraction"); ax.legend(); ax.grid(True, alpha=0.3)

# Te profiles at selected times
ax = axes[1, 0]
rho = traj["rho"]
n_t = len(t)
if len(traj["T_e"]) > 0:
    for idx in [0, n_t // 2, n_t - 1]:
        if idx < len(traj["T_e"]):
            ax.plot(rho, traj["T_e"][idx], label=f"t={t[idx]:.1f}s")
ax.set_xlabel("rho"); ax.set_ylabel("Te (keV)")
ax.set_title("Electron Temperature Profiles"); ax.legend(); ax.grid(True, alpha=0.3)

# ne profiles at selected times
ax = axes[1, 1]
if len(traj["n_e"]) > 0:
    for idx in [0, n_t // 2, n_t - 1]:
        if idx < len(traj["n_e"]):
            ax.plot(rho, traj["n_e"][idx] * 1e-20, label=f"t={t[idx]:.1f}s")
ax.set_xlabel("rho"); ax.set_ylabel("ne (10^20 m^-3)")
ax.set_title("Electron Density Profiles"); ax.legend(); ax.grid(True, alpha=0.3)

# betap vs time
ax = axes[1, 2]
ax.plot(t, traj["betap"], "k-o", markersize=4)
ax.set_xlabel("Time (s)"); ax.set_ylabel("betap")
ax.set_title("Poloidal Beta (TORAX -> FreeGSNKE)"); ax.grid(True, alpha=0.3)

fig.suptitle("Normal Shot \u2014 Sequential Coupling (10s)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Scenario 2: Density-Limit Disruption

Ramp nbar from 0.85 to ~1.34 fGW starting at t=3s. Expect f_GW to cross the 0.95 trigger.

In [ ]:
def density_ramp(t):
    """Ramp nbar from 0.85 to ~1.34 fGW starting at t=3s."""
    if t < 3.0:
        return 0.85
    return 0.85 + 0.07 * (t - 3.0)

print("Running density-limit disruption scenario...")
print("Density ramp: nbar = 0.85 -> ~1.34 fGW starting at t=3s\n")

trajectory_density = run_sequential_coupling(
    eq_solver,
    t_end=10.0,
    dt_couple=1.0,
    Ip_waveform=lambda t: 15e6,
    P_heat_waveform=lambda t: 33e6,
    nbar_waveform=density_ramp,
    transport_model="constant",
    verbose=True,
)

if trajectory_density["disruption_time"]:
    print(f"\nDisruption at t={trajectory_density['disruption_time']:.1f}s "
          f"via {trajectory_density['trigger']}")
else:
    print("\nNo disruption triggered \u2014 consider increasing ramp rate")

## Scenario 3: q95 Disruption (Ip Ramp-Down)

Ramp Ip from 15 MA down to 8 MA starting at t=3s (1 MA/s). Expect q95 to drop below 2.2.

In [ ]:
def Ip_rampdown(t):
    """Ramp Ip from 15 MA down to 8 MA starting at t=3s."""
    if t < 3.0:
        return 15e6
    return max(8e6, 15e6 - 1e6 * (t - 3.0))

print("Running q95 disruption scenario (Ip ramp-down)...")
print("Ip ramp: 15 MA -> 8 MA starting at t=3s (1 MA/s)\n")

trajectory_q95 = run_sequential_coupling(
    eq_solver,
    t_end=10.0,
    dt_couple=1.0,
    Ip_waveform=Ip_rampdown,
    P_heat_waveform=lambda t: 33e6,
    nbar_waveform=lambda t: 0.85,
    transport_model="constant",
    verbose=True,
)

if trajectory_q95["disruption_time"]:
    print(f"\nDisruption at t={trajectory_q95['disruption_time']:.1f}s "
          f"via {trajectory_q95['trigger']}")
else:
    print("\nNo disruption triggered \u2014 q95 may not have dropped below 2.2")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

scenarios = [
    ("Normal", trajectory_normal, "b"),
    ("Density limit", trajectory_density, "r"),
    ("q95 (Ip ramp)", trajectory_q95, "g"),
]

# betaN
ax = axes[0, 0]
for name, traj, color in scenarios:
    ax.plot(traj["time"], traj["betaN"], f"{color}-o", markersize=3, label=name)
    if traj["disruption_time"]:
        ax.axvline(traj["disruption_time"], color=color, linestyle=":", alpha=0.5)
ax.axhline(TRIGGERS["betaN"], color="k", linestyle="--", alpha=0.5, label="limit")
ax.set_xlabel("Time (s)"); ax.set_ylabel("betaN"); ax.set_title("betaN")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# q95
ax = axes[0, 1]
for name, traj, color in scenarios:
    ax.plot(traj["time"], traj["q95"], f"{color}-o", markersize=3, label=name)
    if traj["disruption_time"]:
        ax.axvline(traj["disruption_time"], color=color, linestyle=":", alpha=0.5)
ax.axhline(TRIGGERS["q95"], color="k", linestyle="--", alpha=0.5, label="limit")
ax.set_xlabel("Time (s)"); ax.set_ylabel("q95"); ax.set_title("q95")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# f_GW
ax = axes[0, 2]
for name, traj, color in scenarios:
    ax.plot(traj["time"], traj["f_GW"], f"{color}-o", markersize=3, label=name)
    if traj["disruption_time"]:
        ax.axvline(traj["disruption_time"], color=color, linestyle=":", alpha=0.5)
ax.axhline(TRIGGERS["f_GW"], color="k", linestyle="--", alpha=0.5, label="limit")
ax.set_xlabel("Time (s)"); ax.set_ylabel("f_GW"); ax.set_title("Greenwald Fraction")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Ip
ax = axes[1, 0]
for name, traj, color in scenarios:
    ax.plot(traj["time"], np.array(traj["Ip"]) * 1e-6, f"{color}-o",
            markersize=3, label=name)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Ip (MA)"); ax.set_title("Plasma Current")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# betap
ax = axes[1, 1]
for name, traj, color in scenarios:
    ax.plot(traj["time"], traj["betap"], f"{color}-o", markersize=3, label=name)
ax.set_xlabel("Time (s)"); ax.set_ylabel("betap"); ax.set_title("Poloidal Beta")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# W_thermal
ax = axes[1, 2]
for name, traj, color in scenarios:
    ax.plot(traj["time"], np.array(traj["W_thermal"]) * 1e-6, f"{color}-o",
            markersize=3, label=name)
ax.set_xlabel("Time (s)"); ax.set_ylabel("W_th (MJ)")
ax.set_title("Thermal Energy")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig.suptitle("Disruption Scenario Comparison \u2014 Sequential Coupling",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## DREAM Handoff Preparation

Extract the pre-disruption plasma state from triggered shots, formatted for DREAM input.

In [ ]:
def prepare_dream_handoff(trajectory):
    """Extract pre-disruption state for DREAM input."""
    if not trajectory["disruption_time"]:
        print("No disruption \u2014 nothing to hand off to DREAM.")
        return None

    prof = trajectory["pre_disruption_profiles"]
    rho = trajectory["rho"]

    handoff = {
        "trigger_type": trajectory["trigger"],
        "trigger_time": trajectory["disruption_time"],
        "rho": rho,
        "Te_keV": np.interp(rho, prof["rho_Te"], prof["Te"]),
        "Ti_keV": np.interp(rho, prof["rho_Ti"], prof["Ti"]),
        "ne_m3": np.interp(rho, prof["rho_ne"], prof["ne"]),
        "j_A_m2": np.interp(rho, prof["rho_j"], prof["j"]),
        "q_profile": np.interp(rho, prof["rho_q"], prof["q"]),
        "Ip_A": trajectory["Ip"][-1],
        "W_thermal_J": prof["W_thermal"],
        "geqdsk_dir": trajectory["geqdsk_dir"],
    }

    print(f"DREAM handoff prepared:")
    print(f"  Trigger: {handoff['trigger_type']} at t={handoff['trigger_time']:.1f}s")
    print(f"  Ip = {handoff['Ip_A']*1e-6:.2f} MA")
    print(f"  Te(0) = {handoff['Te_keV'][0]:.1f} keV")
    print(f"  ne(0) = {handoff['ne_m3'][0]:.2e} m^-3")
    print(f"  W_th = {handoff['W_thermal_J']*1e-6:.1f} MJ")
    return handoff


for name, traj in [("Density limit", trajectory_density),
                    ("q95 (Ip ramp)", trajectory_q95)]:
    print(f"\n{'=' * 40}")
    print(f"  {name}")
    print(f"{'=' * 40}")
    handoff = prepare_dream_handoff(traj)

In [ ]:
# Save results
import pickle

# Uncomment to mount Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')

results = {
    "normal": trajectory_normal,
    "density_limit": trajectory_density,
    "q95_disruption": trajectory_q95,
}

save_path = "/content/sequential_coupling_results.pkl"
with open(save_path, "wb") as f:
    pickle.dump(results, f)
print(f"Results saved to {save_path}")

# Uncomment to copy to Drive:
# import shutil
# drive_path = "/content/drive/MyDrive/tokamak_sim/sequential_coupling_results.pkl"
# os.makedirs(os.path.dirname(drive_path), exist_ok=True)
# shutil.copy(save_path, drive_path)
# print(f"Copied to Google Drive: {drive_path}")